# 🏋️ PulseFit · Notebook 03
## Silver & Gold: clean it, connect it, model it

| 01 Set up | 02 Bronze | 03 Silver & Gold | 04 Answer the CEO | 05 Superpowers |
|:-:|:-:|:-:|:-:|:-:|
| ✅ done | ✅ done | 📍 **you are here** | ⬜ | ⬜ |

### Where we are
Bronze holds everything, exactly as the sources sent it. But you still can't answer the CEO from bronze:
the data is messy, and the three systems don't line up.

In this notebook you do the **T** of ELT, entirely in SQL, inside Snowflake:
* **SILVER**: clean, de-duplicate, connect the systems, keep history.
* **GOLD**: a **star schema**, the model analysts query (lecture: *"Dimensional modelling: the star schema"*).

## Step 0: Set the context
> If you see *"does not exist or not authorized"*, run notebooks **01** and **02** first.

In [ ]:
%%sql -r context
-- Every notebook starts by telling Snowflake WHO we are and WHERE we work
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE PULSEFIT_WH;
USE DATABASE PULSEFIT_DW;
USE SCHEMA SILVER;

SELECT CURRENT_ROLE()      AS my_role,
       CURRENT_WAREHOUSE() AS my_warehouse,
       CURRENT_DATABASE()  AS my_database,
       CURRENT_SCHEMA()    AS my_schema;

## Step 1: Profile the data first: what's wrong with it?
Never transform data you haven't looked at. This one query checks bronze for five typical problems:

In [ ]:
%%sql -r data_problems
SELECT 'Duplicate check-in events (same event sent twice)' AS problem,
       COUNT(*) - COUNT(DISTINCT payload:event_id)         AS how_many
FROM BRONZE.CHECKIN_EVENTS
UNION ALL
SELECT 'Scans from cards that belong to no member',
       COUNT(*)
FROM BRONZE.CHECKIN_EVENTS
WHERE payload:card_id::STRING NOT IN (SELECT card_id FROM BRONZE.MEMBER_SNAPSHOTS)
UNION ALL
SELECT 'Members whose email has capitals or extra spaces',
       COUNT(DISTINCT member_id)
FROM BRONZE.MEMBER_SNAPSHOTS
WHERE email <> LOWER(TRIM(email))
UNION ALL
SELECT 'Check-ins on a different day in UTC than in Sri Lanka',
       COUNT(*)
FROM BRONZE.CHECKIN_EVENTS
WHERE TO_DATE(payload:event_ts::TIMESTAMP_NTZ)
   <> TO_DATE(CONVERT_TIMEZONE('UTC', 'Asia/Colombo', payload:event_ts::TIMESTAMP_NTZ))
UNION ALL
SELECT 'Prices stored as text, like "8,000"',
       COUNT(*)
FROM BRONZE.PRICE_LIST
WHERE monthly_fee LIKE '%,%';

👀 **What each problem means**
| Problem | Why it happens | How silver fixes it |
|---|---|---|
| Duplicate events | a scanner loses Wi-Fi and re-sends the same event | keep one row per `event_id` |
| Cards with no member | guest passes and day visitors | keep member check-ins only |
| Messy emails | people type them by hand in the app | `LOWER(TRIM(email))` |
| Different day in UTC | scanners use UTC; Sri Lanka is UTC+5:30, so a 5:15 am workout is 11:45 pm *the day before* in UTC | convert to `Asia/Colombo` time |
| Prices as text | Excel export | convert to numbers |

🕵️ **Mystery solved:** that `checkins_2025-09` file holds early-morning visits on **1 October** (Sri Lanka time),
which were still **30 September** in UTC.

This is what the lecture meant by **integrated**: *one ID, one definition, one timezone and currency.*

## Step 2: Silver: clubs and prices
Give every column a proper type. For prices, each row also gets a **validity window**: `VALID_FROM` (when this price
started) and `VALID_TO` (when the next price took over; `9999-12-31` means *still valid today*).

In [ ]:
%%sql -r silver_clubs_and_prices
CREATE OR REPLACE TABLE SILVER.CLUBS AS
SELECT club_id::INT    AS club_id,
       club_code,
       club_name,
       city,
       province,
       opened_on::DATE AS opened_on
FROM BRONZE.CLUBS;

CREATE OR REPLACE TABLE SILVER.PLAN_PRICES AS
SELECT UPPER(TRIM(plan_code))                    AS plan_code,
       TRIM(plan_name)                           AS plan_name,
       REPLACE(monthly_fee, ',', '')::NUMBER(10) AS monthly_fee_lkr,
       TO_DATE(effective_from, 'DD/MM/YYYY')     AS valid_from,
       COALESCE(LEAD(valid_from) OVER (PARTITION BY plan_code ORDER BY valid_from),
                '9999-12-31'::DATE)              AS valid_to
FROM BRONZE.PRICE_LIST;

SELECT * FROM SILVER.PLAN_PRICES ORDER BY plan_code, valid_from;

👀 Prices are now numbers and dates are real dates. **STANDARD** has two rows: LKR 8,000 until 1 April 2026,
then LKR 9,500. So you can ask *"what did Standard cost on 15 March?"* That's the **time-variant** property of a
warehouse from the lecture.

## Step 3: Silver: members, and their history (SCD Type 2)
The Member App only knows **today's** truth. When Nimal moved to Colombo, the app simply **overwrote** his home club.
That's **SCD Type 1**: history lost.

But the app sends us an export every month, so we can **rebuild the history**. Compare each month with the
previous one; whenever something we care about changes (home club, plan or status), start a **new version** of
the member (lecture: *"History matters: SCD Type 1 vs Type 2"*):

```
 Monthly exports (bronze)              MEMBER_HISTORY (silver, SCD Type 2)
 2026-01-01  Nimal  Kandy    ─┐
 2026-02-01  Nimal  Kandy    ─┴──►   Nimal  Kandy    valid 2024-02-10 → 2026-03-01
 2026-03-01  Nimal  Colombo  ─┐
 2026-04-01  Nimal  Colombo  ─┴──►   Nimal  Colombo  valid 2026-03-01 → 9999-12-31  (current)
```
First, clean the monthly exports:

In [ ]:
%%sql -r silver_member_snapshots
CREATE OR REPLACE TABLE SILVER.MEMBER_SNAPSHOTS AS
SELECT snapshot_date::DATE        AS snapshot_date,
       member_id::INT             AS member_id,
       card_id,
       INITCAP(TRIM(first_name))  AS first_name,
       INITCAP(TRIM(last_name))   AS last_name,
       LOWER(TRIM(email))         AS email,
       birth_year::INT            AS birth_year,
       home_club_id::INT          AS home_club_id,
       UPPER(plan_code)           AS plan_code,
       status,
       joined_on::DATE            AS joined_on
FROM BRONZE.MEMBER_SNAPSHOTS;

SELECT COUNT(*) AS rows_in_monthly_exports, COUNT(DISTINCT member_id) AS members
FROM SILVER.MEMBER_SNAPSHOTS;

Now build the history. `HASH(...)` makes a fingerprint of the columns we track, `LAG` looks at the previous month,
and `LEAD` finds when the next version starts:

In [ ]:
%%sql -r silver_member_history
CREATE OR REPLACE TABLE SILVER.MEMBER_HISTORY AS
WITH fingerprinted AS (
    SELECT *,
           HASH(home_club_id, plan_code, status) AS row_hash,
           LAG(HASH(home_club_id, plan_code, status))
               OVER (PARTITION BY member_id ORDER BY snapshot_date) AS previous_row_hash
    FROM SILVER.MEMBER_SNAPSHOTS
)
SELECT member_id, card_id, first_name, last_name, email, birth_year,
       home_club_id, plan_code, status, joined_on,
       -- the first version starts when the member joined; later versions start on the export date
       IFF(previous_row_hash IS NULL, LEAST(joined_on, snapshot_date), snapshot_date) AS valid_from,
       COALESCE(LEAD(snapshot_date) OVER (PARTITION BY member_id ORDER BY snapshot_date),
                '9999-12-31'::DATE) AS valid_to,
       valid_to = '9999-12-31'::DATE AS is_current
FROM fingerprinted
-- keep only the months where something changed
WHERE previous_row_hash IS NULL
   OR row_hash <> previous_row_hash;

SELECT COUNT(*) AS member_versions, COUNT(DISTINCT member_id) AS members
FROM SILVER.MEMBER_HISTORY;

👀 21,800 monthly rows shrink to about **2,900 versions** for 2,400 members. Most members never changed;
a few hundred moved club, upgraded or cancelled. Let's look at Nimal:

In [ ]:
%%sql -r nimal_history
SELECT member_id, first_name, last_name, home_club_id, plan_code, status, valid_from, valid_to, is_current
FROM SILVER.MEMBER_HISTORY
WHERE member_id = 1001
ORDER BY valid_from;

👀 **Two versions** of Nimal: home club `2` (Kandy) until 1 March 2026, then club `1` (Colombo), which is the
current version. You'll see why this matters in notebook 04.

## Step 4: Silver: check-ins
Four fixes in one statement:
1. **De-duplicate**: `QUALIFY ROW_NUMBER() ... = 1` keeps one row per `event_id`.
2. **Convert time zones**: UTC ➜ `Asia/Colombo`, then take the date. Now visits land on the right day.
3. **Connect the systems**: the scanner knows a `card_id`, the app knows a `member_id`. Joining on the card
   translates one into the other (and drops guest scans, which belong to no member).
4. **Flatten the JSON** into normal, typed columns.

In [ ]:
%%sql -r silver_checkins
CREATE OR REPLACE TABLE SILVER.CHECKINS AS
WITH deduplicated AS (
    SELECT *
    FROM BRONZE.CHECKIN_EVENTS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY payload:event_id ORDER BY _loaded_at) = 1
), member_cards AS (
    SELECT DISTINCT card_id, member_id
    FROM SILVER.MEMBER_SNAPSHOTS
)
SELECT e.payload:event_id::STRING                                      AS checkin_id,
       c.member_id,
       e.payload:card_id::STRING                                       AS card_id,
       e.payload:device.club_code::STRING                              AS club_code,
       e.payload:event_ts::TIMESTAMP_NTZ                               AS checkin_ts_utc,
       CONVERT_TIMEZONE('UTC', 'Asia/Colombo', checkin_ts_utc)         AS checkin_ts_local,
       TO_DATE(checkin_ts_local)                                       AS checkin_date,
       e.payload:session.duration_min::INT                             AS duration_min,
       ARRAY_SIZE(e.payload:session.activities)                        AS activity_count,
       e._source_file
FROM deduplicated e
JOIN member_cards c ON c.card_id = e.payload:card_id::STRING;

SELECT 'Raw events in bronze'                          AS step, COUNT(*) AS row_count FROM BRONZE.CHECKIN_EVENTS
UNION ALL
SELECT 'Unique events (duplicates removed)',                    COUNT(DISTINCT payload:event_id) FROM BRONZE.CHECKIN_EVENTS
UNION ALL
SELECT 'Member check-ins in silver (guest scans removed)',      COUNT(*) FROM SILVER.CHECKINS;

Here are a few early-morning visits where UTC and Sri Lankan time fall on **different days**:

In [ ]:
%%sql -r timezone_examples
SELECT checkin_id, club_code, checkin_ts_utc, checkin_ts_local, checkin_date
FROM SILVER.CHECKINS
WHERE TO_DATE(checkin_ts_utc) <> checkin_date
ORDER BY checkin_ts_local
LIMIT 5;

👀 Without the conversion, every one of these visits would be counted on the **wrong day**, and monthly or
quarterly totals would be slightly wrong. Small errors like this are how *"two reports, two different numbers"*
starts.

## Step 5: Gold: the star schema
Now we model the data for questions. This is the same design as the lecture slide, built for real:

```
                      ┌───────────────────┐
                      │     DIM_DATE      │  when?
                      └─────────┬─────────┘
┌───────────────────┐ ┌─────────┴─────────┐ ┌───────────────────┐
│    DIM_MEMBER     ├─┤    FCT_CHECKIN    ├─┤     DIM_CLUB      │
│ who? (SCD Type 2) │ │ grain: 1 row =    │ │ where?            │
└───────────────────┘ │ 1 visit           │ └───────────────────┘
                      │ duration_min, ... │
                      └─────────┬─────────┘
                      ┌─────────┴─────────┐
                      │     DIM_PLAN      │  which plan, at which price?
                      └───────────────────┘
```
| Term | Meaning here |
|---|---|
| **Grain** | one row in `FCT_CHECKIN` = one visit by one member. Decide this first! |
| **Fact** | numbers you add up: visits (row count), `duration_min`, `activity_count` |
| **Dimension** | the who / what / where / when you slice by: member, plan, club, date |
| **Surrogate key** | `member_key`, `plan_key`, ...: IDs owned by the warehouse. Each *version* of a member gets its own key. |

First the dimensions. `DIM_DATE` is generated, not loaded: one row per day, with handy columns for grouping.

In [ ]:
%%sql -r gold_dim_date
CREATE OR REPLACE TABLE GOLD.DIM_DATE AS
WITH days AS (
    SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, '2025-01-01'::DATE) AS d
    FROM TABLE(GENERATOR(ROWCOUNT => 730))
)
SELECT TO_NUMBER(TO_VARCHAR(d, 'YYYYMMDD'))  AS date_key,
       d                                     AS full_date,
       DAYNAME(d)                            AS day_name,
       DAYOFWEEKISO(d) IN (6, 7)             AS is_weekend,
       DATE_TRUNC('week', d)                 AS week_start,
       TO_VARCHAR(d, 'YYYY-MM')              AS year_month,
       MONTHNAME(d)                          AS month_name,
       YEAR(d) || '-Q' || QUARTER(d)         AS year_quarter,
       YEAR(d)                               AS year_num
FROM days;

SELECT * FROM GOLD.DIM_DATE WHERE full_date BETWEEN '2026-03-29' AND '2026-04-04' ORDER BY full_date;

In [ ]:
%%sql -r gold_dim_club_and_plan
CREATE OR REPLACE TABLE GOLD.DIM_CLUB AS
SELECT ROW_NUMBER() OVER (ORDER BY club_id) AS club_key,
       club_id, club_code, club_name, city, province, opened_on
FROM SILVER.CLUBS;

CREATE OR REPLACE TABLE GOLD.DIM_PLAN AS
SELECT ROW_NUMBER() OVER (ORDER BY plan_code, valid_from) AS plan_key,
       plan_code, plan_name, monthly_fee_lkr, valid_from, valid_to,
       valid_to = '9999-12-31'::DATE AS is_current
FROM SILVER.PLAN_PRICES;

SELECT * FROM GOLD.DIM_PLAN ORDER BY plan_key;

In [ ]:
%%sql -r gold_dim_member
CREATE OR REPLACE TABLE GOLD.DIM_MEMBER AS
SELECT ROW_NUMBER() OVER (ORDER BY h.member_id, h.valid_from) AS member_key,
       h.member_id,
       h.first_name || ' ' || h.last_name AS member_name,
       h.email,
       CASE WHEN 2026 - h.birth_year < 25 THEN '18-24'
            WHEN 2026 - h.birth_year < 35 THEN '25-34'
            WHEN 2026 - h.birth_year < 45 THEN '35-44'
            WHEN 2026 - h.birth_year < 55 THEN '45-54'
            ELSE '55+' END                AS age_band,
       h.joined_on,
       c.club_name                        AS home_club,
       h.plan_code,
       h.status,
       h.valid_from,
       h.valid_to,
       h.is_current
FROM SILVER.MEMBER_HISTORY h
JOIN SILVER.CLUBS c ON c.club_id = h.home_club_id;

SELECT member_key, member_id, member_name, home_club, plan_code, valid_from, valid_to, is_current
FROM GOLD.DIM_MEMBER
WHERE member_id = 1001;

Now the **fact table**. The important trick is the **point-in-time join**: each visit is linked to the version of
the member, and the price of the plan, **that was true on the day of the visit**.

In [ ]:
%%sql -r gold_fct_checkin
CREATE OR REPLACE TABLE GOLD.FCT_CHECKIN AS
SELECT ci.checkin_id,
       d.date_key,
       m.member_key,
       c.club_key,
       p.plan_key,
       ci.checkin_ts_local,
       ci.duration_min,
       ci.activity_count
FROM SILVER.CHECKINS ci
JOIN GOLD.DIM_DATE d
  ON d.full_date = ci.checkin_date
JOIN GOLD.DIM_CLUB c
  ON c.club_code = ci.club_code
-- the member version that was valid on the visit date (SCD Type 2)
JOIN GOLD.DIM_MEMBER m
  ON m.member_id = ci.member_id
 AND ci.checkin_date >= m.valid_from AND ci.checkin_date < m.valid_to
-- the price that was valid on the visit date, for the plan the member had that day
JOIN GOLD.DIM_PLAN p
  ON p.plan_code = m.plan_code
 AND ci.checkin_date >= p.valid_from AND ci.checkin_date < p.valid_to;

SELECT 'DIM_DATE'    AS table_name, COUNT(*) AS row_count FROM GOLD.DIM_DATE
UNION ALL SELECT 'DIM_CLUB',    COUNT(*) FROM GOLD.DIM_CLUB
UNION ALL SELECT 'DIM_PLAN',    COUNT(*) FROM GOLD.DIM_PLAN
UNION ALL SELECT 'DIM_MEMBER',  COUNT(*) FROM GOLD.DIM_MEMBER
UNION ALL SELECT 'FCT_CHECKIN', COUNT(*) FROM GOLD.FCT_CHECKIN;

## Step 6: The quality gate: unit tests for data
Before anyone uses gold, we **prove** it is correct. Each test counts *failures*, so `0` means pass
(lecture: *"Data quality: unit tests for data"*). The tests run **as a gate**: if one fails, the pipeline stops.

In [ ]:
%%sql -r quality_tests
WITH tests AS (
    SELECT 'FCT_CHECKIN: every checkin_id is unique' AS test,
           (SELECT COUNT(*) - COUNT(DISTINCT checkin_id) FROM GOLD.FCT_CHECKIN) AS failures
    UNION ALL
    SELECT 'Reconciliation: every silver check-in reached the fact table',
           (SELECT COUNT(*) FROM SILVER.CHECKINS) - (SELECT COUNT(*) FROM GOLD.FCT_CHECKIN)
    UNION ALL
    SELECT 'DIM_MEMBER: exactly one current version per member',
           (SELECT COUNT(*) FROM (SELECT member_id FROM GOLD.DIM_MEMBER
                                  GROUP BY member_id HAVING COUNT_IF(is_current) <> 1))
    UNION ALL
    SELECT 'DIM_MEMBER: history windows never overlap',
           (SELECT COUNT(*) FROM GOLD.DIM_MEMBER a
            JOIN GOLD.DIM_MEMBER b
              ON a.member_id = b.member_id AND a.member_key < b.member_key
             AND a.valid_from < b.valid_to AND b.valid_from < a.valid_to)
    UNION ALL
    SELECT 'FCT_CHECKIN: visit duration between 1 and 300 minutes',
           (SELECT COUNT(*) FROM GOLD.FCT_CHECKIN WHERE duration_min NOT BETWEEN 1 AND 300)
)
SELECT test, failures, IFF(failures = 0, '✅ PASS', '❌ FAIL') AS result
FROM tests;

The Python cell below is the **gate**. It reads the test results from the SQL cell above (each SQL cell's result
is available in Python under the cell's name, here `quality_tests`) and stops with an error if any test failed.

In [ ]:
results = quality_tests.to_pandas()
failed = results[results["FAILURES"] != 0]

if len(failed) > 0:
    raise Exception("❌ Quality gate FAILED. Do not publish gold:\n" + failed.to_string(index=False))

print(f"✅ All {len(results)} tests passed. Gold is safe to publish. 🎉")

## Step 7: A friendly view for analysts
A **view** is a saved query: it stores **no data** and is always up to date. We give analysts one wide,
easy-to-read view over the star schema. We can then change the tables underneath without breaking their reports.
The lecture called this the **contract**.

In [ ]:
%%sql -r gold_view
CREATE OR REPLACE VIEW GOLD.CHECKIN_DETAILS
    COMMENT = 'One row per visit, with everything an analyst needs'
AS
SELECT f.checkin_id,
       d.full_date        AS visit_date,
       d.week_start,
       d.year_month,
       d.year_quarter,
       d.day_name,
       c.club_name        AS visited_club,
       m.member_id,
       m.member_name,
       m.age_band,
       m.home_club,
       m.plan_code,
       p.monthly_fee_lkr  AS fee_at_visit_lkr,
       f.checkin_ts_local,
       f.duration_min,
       f.activity_count
FROM GOLD.FCT_CHECKIN f
JOIN GOLD.DIM_DATE   d ON d.date_key   = f.date_key
JOIN GOLD.DIM_CLUB   c ON c.club_key   = f.club_key
JOIN GOLD.DIM_MEMBER m ON m.member_key = f.member_key
JOIN GOLD.DIM_PLAN   p ON p.plan_key   = f.plan_key;

SELECT * FROM GOLD.CHECKIN_DETAILS ORDER BY checkin_ts_local LIMIT 10;

## ✅ What you did
* **Profiled** bronze and found five real-world data problems.
* Built **silver**: typed, cleaned, de-duplicated, time zones fixed, and **systems connected** (card ➜ member).
* Rebuilt member **history (SCD Type 2)** from monthly snapshots.
* Built a **star schema** in gold, with **point-in-time joins** to the right member version and price.
* Ran **data-quality tests as a gate**, and published a **view** as the analysts' contract.

All of it was plain SQL running inside the warehouse. That's **ELT**.

## 🤔 Think about it
1. Our history is built from **monthly** exports. If Nimal had moved on **10 March**, what `VALID_FROM` would our
   history show? Which visits would be linked to the wrong club, and how could PulseFit capture changes more precisely?
2. Why does `FCT_CHECKIN` store `member_key` (one per *version*) instead of simply `member_id`?

## ➡️ Next
Open **`04_Answer_the_CEO`**. Time to answer the question!